# Cross-dataset OOD Detection Experiments

In the following we demonstrate how to reproduce the cross-dataset OOD detection experiments.

We use the CIFAR-10-trained model as an example and reproduce the results for the CIFAR-10 --> SVHN and CIFAR-10 --> CelebA setups.

In [ ]:
import numpy as np
import pandas as pd

from sitn.aggregators import KDE, MaxQuantile
from sitn.metrics import bootstrap_auroc
from sitn.utils import construct_results_path, submit_eval_jobs, submit_train_jobs

## Train Flow Matching Model

In [ ]:
# Training config
train_cfg = {"dataset_name": "cifar10"}

In [ ]:
# Submit slurm job for training (or use CLI instead)
submit_train_jobs([train_cfg])

Alternatively, via CLI:

`uv run sitn-train --dataset_name cifar10`

## Evaluate Likelihoods

In [ ]:
# Evaluation configs for train/val/test on CIFAR-10
# Train and val are used to fit aggregation models like the
# max-quantile approach for SITN or the KDE for the DoSE baseline.
# Test will be used as ID evaluation data.
eval_cfg_train = {"config": train_cfg, "split_pick": "train"}
eval_cfg_val = {"config": train_cfg, "split_pick": "val"}
eval_cfg_test = {"config": train_cfg, "split_pick": "test"}

# Evaluation configs for the two OOD evaluation datasets
eval_cfg_svhn = {"config": train_cfg, "eval_dataset_name": "svhn", "split_pick": "test"}
eval_cfg_celeba = {"config": train_cfg, "eval_dataset_name": "celeba", "split_pick": "test"}

In [ ]:
# Submit slurm jobs for evaluations (or use CLI instead)
# These jobs should only be submitted after training has completed.
# Note: at the end of training, evaluations are automatically run
# on the train, val, and test splits of the training dataset, so
# no additional evaluation jobs need to be submitted for them.
submit_eval_jobs([eval_cfg_svhn, eval_cfg_celeba])

Alternatively, via CLI:

`uv run sitn-eval /path/to/training_cfg --eval_dataset_name svhn --split_pick test`

`uv run sitn-eval /path/to/training_cfg --eval_dataset_name celeba --split_pick test`

Note that the training config is created and saved in the output folder when a model is trained.

## Fit OOD Methods

In [ ]:
# Load train and val ID predictions
id_train_preds = pd.read_csv(construct_results_path(**eval_cfg_train, result_type="predictions"))
id_val_preds = pd.read_csv(construct_results_path(**eval_cfg_val, result_type="predictions"))

# Compute entropy estimate for typicality
entropy_estimate = np.mean(id_train_preds["log_likelihood"])

# Fit DoSE
dose = KDE(features=["log_likelihood", "source_log_likelihood", "log_determinant"])
dose.fit(id_train_preds, subsample=10000)

# Fit SITN
sitn = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
sitn.fit(id_val_preds)

## Evaluate OOD Detection Performance

In [ ]:
# Metric configurations
metrics = {
    "log_likelihood": {"label": "Log Likelihood", "higher_is_ood": False},
    "typicality": {"label": "Typicality", "higher_is_ood": True},
    "dose": {"label": "DoSE", "higher_is_ood": False},
    "sitn": {"label": "SITN", "higher_is_ood": True},
}

# Load ID test predictions
id_preds = pd.read_csv(construct_results_path(**eval_cfg_test, result_type="predictions"))
id_preds["train_dataset"] = eval_cfg_test["config"]["dataset_name"]
id_preds["eval_dataset"] = eval_cfg_test["config"]["dataset_name"]

results = []
for eval_cfg_ood in [eval_cfg_svhn, eval_cfg_celeba]:
    # Load OOD test predictions
    ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
    ood_preds["train_dataset"] = eval_cfg_ood["config"]["dataset_name"]
    ood_preds["eval_dataset"] = eval_cfg_ood["eval_dataset_name"]

    # Combine ID and OOD predictions
    preds = pd.concat([id_preds.copy(), ood_preds], ignore_index=True)

    # Add baseline and SITN scores
    preds["typicality"] = (preds["log_likelihood"] - entropy_estimate).abs()
    preds["dose"] = dose.score(preds)
    preds["sitn"] = sitn.score(preds)

    # Compute AUROC with bootstrapped CIs for each method
    y_true = (preds["eval_dataset"] != preds["train_dataset"]).astype(int)
    for col, meta in metrics.items():
        scores = preds[col].copy()
        if not meta["higher_is_ood"]:
            scores = -scores

        auroc, ci_lo, ci_hi = bootstrap_auroc(y_true, scores)
        results.append(
            {
                "ood_dataset": eval_cfg_ood["eval_dataset_name"],
                "metric": meta["label"],
                "AUROC": auroc,
                "CI_lo": ci_lo,
                "CI_hi": ci_hi,
            }
        )

results = pd.DataFrame(results).set_index(["ood_dataset", "metric"])
results

AUROC     CI_lo     CI_hi
ood_dataset metric                                      
svhn        Log Likelihood  0.104947  0.100382  0.109669
            Typicality      0.790203  0.784629  0.795829
            DoSE            0.801911  0.795948  0.807798
            SITN            0.946121  0.943365  0.948894
celeba      Log Likelihood  0.495693  0.488294  0.503064
            Typicality      0.420800  0.413666  0.427705
            DoSE            0.431566  0.424546  0.438502
            SITN            0.671056  0.664537  0.677723